# Phase 4 — Physical & Environmental Suitability
**Ontario Distribution-Connected Solar Siting | 10+ MW Projects**

Compute all parcel-level physical, environmental, and suitability attributes
required as inputs to Phase 5 scoring and triage. Unit of analysis: **parcels**
(from `developable_parcels_{slug}`).

**Input**
- `analysis.developable_parcels_{county_slug}` — Phase 3 + geocoding output (one row per parcel)

**Steps**
1. Load Phase 3 parcels → `parcels_phase4_{slug}` (working copy, enriched step by step)
2. Slope — mean slope (%) over buildable_geom via DEM raster zonal statistics
3. PVout — mean irradiance (kWh/kWp/yr) over buildable_geom via PVout raster
4. Noise receptors — building count within 1 km of parcel_geom
5. Compactness — Polsby-Popper ratio on buildable_geom
6. CA authority overlap — regulated area / buildable area ratio
7. Environmental flags + ECI — 8 binary flags, weighted sum (max 19)
8. SARO species assessment — END/THR/SC counts per parcel
9. Triage classification — GREEN / AMBER / RED with structured flags
10. Verification & interactive map

---
Change only `COUNTY_NAME`, `CONSERVATION_AUTH`, `DEM_PATH`, and `PVOUT_PATH` in the Configuration cell to reproduce for any county.

## 0 · Configuration

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# PROJECT PARAMETERS — edit this cell only to run for a different county
# ─────────────────────────────────────────────────────────────────────────────

import os

COUNTY_NAME       = "Peterborough"          # Must match adm_district_township.name_2
CONSERVATION_AUTH = "orca"            # orca (PTB) | rvca (OTT) | utrca (OXF)

# Raster paths — GeoTIFF in EPSG:5321
DEM_PATH   = "/data/rasters/dem_peterborough_5321.tif"    # e.g. "/data/rasters/dem_ontario_5321.tif"
PVOUT_PATH = "data/rasters/pvout_ontario_5321.tif"    # e.g. "/data/rasters/pvout_ontario_5321.tif"

# PostGIS connection
PG_CONN = os.environ["POSTGRES_CONNECTION_STRING"]

# Output schema
OUTPUT_SCHEMA = "analysis"

# CRS
CRS_WGS84   = 4326
CRS_NAD83   = 4269   # NAD83 geographic — bridge CRS
CRS_ONTARIO = 5321   # NAD83(CSRS) / Ontario MNR Lambert — all calculations

# ── Runtime sanity checks ──────────────────────────────────────────────────
import pathlib

_errors = []
if not DEM_PATH:
    _errors.append("DEM_PATH is not set")
elif not pathlib.Path(DEM_PATH).exists():
    _errors.append(f"DEM_PATH not found: {DEM_PATH}")

if not PVOUT_PATH:
    _errors.append("PVOUT_PATH is not set")
elif not pathlib.Path(PVOUT_PATH).exists():
    _errors.append(f"PVOUT_PATH not found: {PVOUT_PATH}")

if _errors:
    for e in _errors:
        print(f"  ⚠️  {e}")
    print("\nSet DEM_PATH and PVOUT_PATH before running Steps 2 and 3.")
else:
    print("  ✅  Raster paths verified.")

print(f"County            : {COUNTY_NAME}")
print(f"Conservation auth : {CONSERVATION_AUTH}")


## 1 · Environment Setup & Utilities

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import MiniMap
from branca.colormap import linear
from sqlalchemy import create_engine, text

engine = create_engine(PG_CONN)


def read_postgis(sql: str, geom_col: str = "geom") -> gpd.GeoDataFrame:
    """Execute a PostGIS query and return a WGS84 GeoDataFrame."""
    gdf = gpd.read_postgis(sql, engine, geom_col=geom_col)
    if gdf.crs is None:
        gdf = gdf.set_crs(epsg=CRS_WGS84)
    return gdf.to_crs(epsg=CRS_WGS84)


def save_to_postgis(gdf: gpd.GeoDataFrame, table: str, label: str) -> None:
    """Write a GeoDataFrame to PostGIS in EPSG:5321 and create a GiST spatial index."""
    geom_col = gdf.geometry.name
    gdf.to_crs(epsg=CRS_ONTARIO).to_postgis(
        name=table,
        con=engine,
        schema=OUTPUT_SCHEMA,
        if_exists="replace",
        index=False,
        chunksize=500
    )
    with engine.connect() as conn:
        conn.execute(text(
            f"CREATE INDEX IF NOT EXISTS {table}_geom_idx "
            f"ON {OUTPUT_SCHEMA}.{table} USING GIST({geom_col})"
        ))
        conn.commit()
    print(f"  {label:40s} → {OUTPUT_SCHEMA}.{table} "
          f"({len(gdf):,} rows, EPSG:{CRS_ONTARIO}, GiST index)")


def update_parcels_phase4(df_updates: pd.DataFrame, join_col: str = "parcel_id") -> None:
    """
    Merge scalar columns from df_updates into analysis.parcels_phase4_{slug} in-place.
    Reads the current table, left-joins df_updates, and overwrites the table.
    The geometry column from the existing table is preserved.
    """
    table = f"parcels_phase4_{county_slug}"
    gdf_current = read_postgis(
        f"SELECT * FROM {OUTPUT_SCHEMA}.{table}",
        geom_col="buildable_geom"
    )
    # Drop any columns that are already present in df_updates (will be replaced)
    cols_to_drop = [c for c in df_updates.columns if c != join_col and c in gdf_current.columns]
    gdf_current = gdf_current.drop(columns=cols_to_drop)
    gdf_merged = gdf_current.merge(df_updates, on=join_col, how="left")
    save_to_postgis(gdf_merged, table, f"Update parcels_phase4 (+{list(df_updates.columns)})")


county_slug = COUNTY_NAME.lower().replace(" ", "_")

with engine.connect() as conn:
    print("PostGIS:", conn.execute(text("SELECT PostGIS_Full_Version()")).scalar())
print(f"County : {COUNTY_NAME} (slug: {county_slug})")


---
## Step 1 — Load Phase 3 Parcels

Load `developable_parcels_{slug}` (Phase 3 + geocoding output) and save a
working copy as `parcels_phase4_{slug}`. This table will be enriched step by
step — each subsequent step reads from `analysis.parcels_phase4_{slug}`.

Two geometry columns are carried forward:
- **`buildable_geom`** — final developable area polygon (land-use clipped, building-subtracted, parcel-intersected)
- **`parcel_geom`** — full original parcel boundary (for environmental flags and noise receptor counts)

`buildable_area_ac` is computed from `buildable_geom` in EPSG:5321.

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# developable_parcels has a single geom column (parcel outline).
# developable_area_parcels has the clipped buildable geometry.
# We join them to get both geometry columns on one row per parcel.
gdf_parcels = read_postgis(f"""
SELECT
    dp.parcel_id,
    dp.solar_parcel_uid,
    dp.main_address,
    dp.parcel_acre,
    -- Buildable geometry: intersected, land-use filtered, building-subtracted area
    ST_Transform(dap_agg.buildable_geom, {CRS_WGS84})  AS buildable_geom,
    -- Full parcel boundary
    ST_Transform(dp.geom, {CRS_WGS84})                 AS parcel_geom,
    -- Buildable area in acres (EPSG:5321 metres → acres)
    ROUND((ST_Area(dap_agg.buildable_geom) / 4046.856)::numeric, 3) AS buildable_area_ac,
    -- Grid attributes from Phase 1 (STRING_AGG may give multiple; take first numeric value)
    CAST(SPLIT_PART(dap_agg.grid_capacity_mw, ', ', 1) AS double precision) AS grid_mw,
    CAST(SPLIT_PART(dap_agg.voltage_3ph, ', ', 1)      AS double precision) AS feeder_voltage
FROM {OUTPUT_SCHEMA}.developable_parcels_{county_slug} dp
JOIN (
    -- Aggregate multiple buildable fragments per parcel into one union geometry
    SELECT
        parcel_id,
        ST_Union(ST_Transform(geom, {CRS_ONTARIO}))      AS buildable_geom,
        STRING_AGG(DISTINCT CAST(grid_capacity_mw AS text), ', ') AS grid_capacity_mw,
        STRING_AGG(DISTINCT CAST(voltage_3ph AS text), ', ')      AS voltage_3ph
    FROM {OUTPUT_SCHEMA}.developable_area_parcels_{county_slug}
    GROUP BY parcel_id
) dap_agg ON dp.parcel_id = dap_agg.parcel_id
WHERE dap_agg.buildable_geom IS NOT NULL
ORDER BY buildable_area_ac DESC
""", geom_col="parcel_geom")

# Make buildable_geom accessible as a plain column (not the active geometry)
gdf_parcels["buildable_geom"] = gpd.GeoSeries.from_wkt(
    gdf_parcels["buildable_geom"].apply(lambda g: g.wkt if g is not None else None),
    crs=CRS_WGS84
)

print(f"Step 1 — Phase 3 parcels loaded for '{COUNTY_NAME}':")
print(f"  Parcel count    : {len(gdf_parcels):,}")
print(f"  Buildable area  : {gdf_parcels['buildable_area_ac'].sum():,.1f} ac total")
print(f"  Columns         : {list(gdf_parcels.columns)}")

save_to_postgis(gdf_parcels, f"parcels_phase4_{county_slug}", "Step 1 — Phase 4 working copy")


---
## Step 2 — Slope

Compute mean slope (%) over **buildable_geom** using Rasterio zonal statistics
on the DEM raster (EPSG:5321 GeoTIFF). Slope is derived from elevation change
per pixel distance; the result is stored as `slope_mean` in percent.

Process:
1. For each parcel, reproject `buildable_geom` to the raster's native CRS
2. Clip the DEM to the parcel's bounding box + reproject geometry mask
3. Compute per-pixel slope from the elevation values (rise/run → arctan → %)
4. Average the valid (non-nodata) pixels within the mask

Requires: `rasterio`, `numpy`, `scipy` (for gradient-based slope)

In [ ]:
import rasterio
import rasterio.mask
import rasterio.transform
from rasterio.crs import CRS as RioCRS
from rasterio.warp import transform_geom
from shapely.geometry import mapping, shape

county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Load current Phase 4 working table (buildable_geom is in CRS_WGS84 before reprojection)
gdf_p4 = read_postgis(
    f"SELECT parcel_id, buildable_geom FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}",
    geom_col="buildable_geom"
)

def _compute_mean_slope(geom_wgs84, dem_path: str) -> float | None:
    """
    Compute mean slope (%) over a geometry using a DEM raster.
    geom_wgs84: shapely geometry in WGS84
    dem_path: path to GeoTIFF (any CRS — reprojected on-the-fly)
    Returns mean slope in %, or None if no valid pixels.
    """
    try:
        with rasterio.open(dem_path) as src:
            raster_crs = src.crs.to_epsg()
            # Reproject geometry to raster CRS
            geom_native = shape(
                transform_geom(
                    f"EPSG:{CRS_WGS84}", f"EPSG:{raster_crs}",
                    mapping(geom_wgs84)
                )
            )
            # Mask raster to geometry
            out_image, out_transform = rasterio.mask.mask(
                src, [mapping(geom_native)], crop=True, nodata=src.nodata, filled=True
            )
            elev = out_image[0].astype(float)
            nodata = src.nodata
            if nodata is not None:
                elev[elev == nodata] = np.nan

            if np.all(np.isnan(elev)) or elev.size == 0:
                return None

            # Pixel resolution in metres (assumes projected CRS)
            res_x = abs(out_transform.a)
            res_y = abs(out_transform.e)

            # Gradient-based slope: rise/run in x and y directions
            dy, dx = np.gradient(elev, res_y, res_x)
            slope_pct = np.sqrt(dx**2 + dy**2) * 100  # percent slope

            # Mask nodata positions
            valid = ~np.isnan(elev)
            if valid.sum() == 0:
                return None
            return float(np.nanmean(slope_pct[valid]))
    except Exception as exc:
        print(f"    slope error for parcel: {exc}")
        return None


slope_results = []
for _, row in gdf_p4.iterrows():
    slope_val = _compute_mean_slope(row.geometry, DEM_PATH) if DEM_PATH else None
    slope_results.append({"parcel_id": row["parcel_id"], "slope_mean": slope_val})

df_slope = pd.DataFrame(slope_results)
null_count = df_slope["slope_mean"].isna().sum()
print(f"Step 2 — Slope results for '{COUNTY_NAME}':")
print(f"  Parcels with slope data : {len(df_slope) - null_count:,}")
print(f"  Parcels with NULL slope : {null_count:,}")
print(f"  Mean slope (county)     : {df_slope['slope_mean'].mean():.2f} %")

update_parcels_phase4(df_slope)


---
## Step 3 — PVout (Irradiance)

Compute mean annual PVout (kWh/kWp/year) over **buildable_geom** using Rasterio
zonal statistics on the PVout raster (EPSG:5321 GeoTIFF).

Higher PVout = more solar energy yield per installed kWp. Stored as `pvout_mean`.

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

gdf_p4 = read_postgis(
    f"SELECT parcel_id, buildable_geom FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}",
    geom_col="buildable_geom"
)

def _compute_mean_raster_value(geom_wgs84, raster_path: str) -> float | None:
    """
    Compute mean raster value over a geometry.
    Handles any CRS by reprojecting geometry to match the raster.
    Returns float or None if no valid pixels.
    """
    try:
        with rasterio.open(raster_path) as src:
            raster_crs_epsg = src.crs.to_epsg()
            geom_native = shape(
                transform_geom(
                    f"EPSG:{CRS_WGS84}", f"EPSG:{raster_crs_epsg}",
                    mapping(geom_wgs84)
                )
            )
            out_image, _ = rasterio.mask.mask(
                src, [mapping(geom_native)], crop=True, nodata=src.nodata, filled=True
            )
            vals = out_image[0].astype(float)
            if src.nodata is not None:
                vals[vals == src.nodata] = np.nan
            if np.all(np.isnan(vals)) or vals.size == 0:
                return None
            return float(np.nanmean(vals))
    except Exception as exc:
        print(f"    raster error for parcel: {exc}")
        return None


pvout_results = []
for _, row in gdf_p4.iterrows():
    pvout_val = _compute_mean_raster_value(row.geometry, PVOUT_PATH) if PVOUT_PATH else None
    pvout_results.append({"parcel_id": row["parcel_id"], "pvout_mean": pvout_val})

df_pvout = pd.DataFrame(pvout_results)
null_count = df_pvout["pvout_mean"].isna().sum()
print(f"Step 3 — PVout results for '{COUNTY_NAME}':")
print(f"  Parcels with PVout data : {len(df_pvout) - null_count:,}")
print(f"  Parcels with NULL PVout : {null_count:,}")
print(f"  Mean PVout (county)     : {df_pvout['pvout_mean'].mean():.1f} kWh/kWp/yr")

update_parcels_phase4(df_pvout)


---
## Step 4 — Noise Receptors

Count buildings within **1,000 m** of `parcel_geom` (full parcel boundary, not
buildable area only). Uses `ST_DWithin` on EPSG:5321 geometries. The result is
stored as `noise_count_1km` (integer, nullable).

**NULL treatment**: A NULL result (no buildings table for this county, or no
rows returned) is stored as NULL — not 0. Phase 5 will treat NULL as zero
receptors = maximum score. This prevents data-gap counties from being penalised.

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Check whether buildings table exists for this county
with engine.connect() as conn:
    has_buildings = conn.execute(text(
        "SELECT EXISTS (SELECT 1 FROM information_schema.tables "
        "WHERE table_schema = 'cadastre' AND table_name = :tbl)"
    ), {"tbl": f"buildings_{county_slug}"}).scalar()

print(f"Buildings table 'cadastre.buildings_{county_slug}' exists: {has_buildings}")

if has_buildings:
    gdf_noise = read_postgis(f"""
        SELECT
            p.parcel_id,
            COUNT(b.geom)::integer AS noise_count_1km
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        LEFT JOIN cadastre.buildings_{county_slug} b
            ON ST_DWithin(
                    ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                    ST_Transform(b.geom, {CRS_ONTARIO}),
                    1000
               )
        GROUP BY p.parcel_id
    """, geom_col="parcel_geom")
    # Convert COUNT 0 → explicit 0 (not NULL); if no buildings exist, keep as-is
    gdf_noise["noise_count_1km"] = gdf_noise["noise_count_1km"].astype("Int64")
    df_noise = gdf_noise[["parcel_id", "noise_count_1km"]].copy()
else:
    # No buildings layer — store NULL for all parcels (Phase 5 treats as MAX score)
    gdf_p4_ids = read_postgis(
        f"SELECT parcel_id FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}",
        geom_col="buildable_geom"
    )
    df_noise = pd.DataFrame({"parcel_id": gdf_p4_ids["parcel_id"], "noise_count_1km": pd.NA})
    df_noise["noise_count_1km"] = df_noise["noise_count_1km"].astype("Int64")

print(f"Step 4 — Noise receptor results for '{COUNTY_NAME}':")
print(f"  Parcels total           : {len(df_noise):,}")
print(f"  Parcels with NULL       : {df_noise['noise_count_1km'].isna().sum():,}")
print(f"  Max buildings within 1km: {df_noise['noise_count_1km'].max()}")
print(f"  Mean buildings          : {df_noise['noise_count_1km'].mean():.1f}")

update_parcels_phase4(df_noise)


---
## Step 5 — Compactness (Polsby-Popper)

Compute the Polsby-Popper ratio on **buildable_geom**:

$$PP = \frac{4\pi \cdot A}{P^2}$$

where $A$ = area and $P$ = perimeter. A value of 1.0 = perfect circle; values
near 0 = elongated or highly irregular shape. Stored as `pp_ratio` (float, 0–1).

This penalises fragmented or highly irregular buildable areas that are difficult
to develop with standard panel arrays.

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

gdf_pp = read_postgis(f"""
    SELECT
        parcel_id,
        ROUND(
            (4.0 * pi() * ST_Area(ST_Transform(buildable_geom, {CRS_ONTARIO}))) /
            NULLIF(
                ST_Perimeter(ST_Transform(buildable_geom, {CRS_ONTARIO})) *
                ST_Perimeter(ST_Transform(buildable_geom, {CRS_ONTARIO})),
                0
            ),
            6
        ) AS pp_ratio,
        buildable_geom
    FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}
""", geom_col="buildable_geom")

df_pp = gdf_pp[["parcel_id", "pp_ratio"]].copy()
df_pp["pp_ratio"] = pd.to_numeric(df_pp["pp_ratio"], errors="coerce")

print(f"Step 5 — Polsby-Popper compactness for '{COUNTY_NAME}':")
print(f"  Parcels computed : {len(df_pp):,}")
print(f"  Mean PP ratio    : {df_pp['pp_ratio'].mean():.4f}")
print(f"  Min PP ratio     : {df_pp['pp_ratio'].min():.4f}")
print(f"  Max PP ratio     : {df_pp['pp_ratio'].max():.4f}")

update_parcels_phase4(df_pp)


---
## Step 6 — CA Authority Overlap Ratio

Compute the ratio of `buildable_geom` that falls within the Conservation Authority
regulated area for this county:

$$ca\_overlap\_ratio = \frac{ST\_Area(buildable\_geom \cap CA\_regulated\_area)}{ST\_Area(buildable\_geom)}$$

Table used: `rea.{CONSERVATION_AUTH}_regulated_area` (set in config).
Result stored as `ca_overlap_ratio` (float, 0.0–1.0).

This value feeds:
- ECI environmental risk score (flag_ca_regulated, weight 3)
- Triage classification (RED if ≥ 0.60; AMBER if > 0 and < 0.60)

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Check whether CA regulated area table exists
with engine.connect() as conn:
    has_ca = conn.execute(text(
        "SELECT EXISTS (SELECT 1 FROM information_schema.tables "
        "WHERE table_schema = 'rea' AND table_name = :tbl)"
    ), {"tbl": f"{CONSERVATION_AUTH}_regulated_area"}).scalar()

print(f"CA table 'rea.{CONSERVATION_AUTH}_regulated_area' exists: {has_ca}")

if has_ca:
    gdf_ca = read_postgis(f"""
        SELECT
            p.parcel_id,
            COALESCE(
                SUM(
                    ST_Area(
                        ST_Intersection(
                            ST_Transform(p.buildable_geom, {CRS_ONTARIO}),
                            ST_Transform(ca.geom, {CRS_ONTARIO})
                        )
                    )
                ) / NULLIF(ST_Area(ST_Transform(p.buildable_geom, {CRS_ONTARIO})), 0),
                0.0
            ) AS ca_overlap_ratio,
            p.buildable_geom
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        LEFT JOIN rea.{CONSERVATION_AUTH}_regulated_area ca
            ON ST_Intersects(
                ST_Transform(p.buildable_geom, {CRS_ONTARIO}),
                ST_Transform(ca.geom, {CRS_ONTARIO})
               )
        GROUP BY p.parcel_id, p.buildable_geom
    """, geom_col="buildable_geom")

    # Clamp to [0, 1] in case of topology edge cases
    gdf_ca["ca_overlap_ratio"] = gdf_ca["ca_overlap_ratio"].clip(0.0, 1.0)
    df_ca = gdf_ca[["parcel_id", "ca_overlap_ratio"]].copy()
else:
    gdf_p4_ids = read_postgis(
        f"SELECT parcel_id FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}",
        geom_col="buildable_geom"
    )
    df_ca = pd.DataFrame({"parcel_id": gdf_p4_ids["parcel_id"], "ca_overlap_ratio": 0.0})
    print(f"  No CA table found — ca_overlap_ratio set to 0.0 for all parcels")

print(f"Step 6 — CA overlap ratio for '{COUNTY_NAME}':")
print(f"  Parcels with overlap > 0  : {(df_ca['ca_overlap_ratio'] > 0).sum():,}")
print(f"  Parcels with overlap ≥ 0.6: {(df_ca['ca_overlap_ratio'] >= 0.6).sum():,}")
print(f"  Mean overlap ratio        : {df_ca['ca_overlap_ratio'].mean():.4f}")

update_parcels_phase4(df_ca)


---
## Step 7 — Environmental Flags (ECI Inputs)

Compute 8 binary flags via spatial joins to **`parcel_geom`** (full boundary).
Each flag = 1 if the environmental layer intersects the parcel, 0 otherwise.

`flag_ca_regulated` is derived directly from `ca_overlap_ratio > 0` (no separate spatial join).

| Flag | Source | Weight |
|---|---|---|
| `flag_wetland` | `rea.wetland` | 3 |
| `flag_woodland` | `rea.woodland` | 3 |
| `flag_nhs` | `environment.nhs_area` | 2 |
| `flag_saro_prox` | `environment.prov_trk_species` | 2 |
| `flag_waa` | `environment.wildlife_activity_area` | 2 |
| `flag_ca_regulated` | derived from `ca_overlap_ratio > 0` | 3 |
| `flag_flood` | `environment.flood_hazard` | 3 |
| `flag_cli` | `environment.cli` (class 1 or 2) | 1 |

`eci_raw` = weighted sum (max possible = 19). Also stored as `env_flags` JSONB.

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# Helper: check existence of a table
def _table_exists(schema: str, table: str) -> bool:
    with engine.connect() as conn:
        return conn.execute(text(
            "SELECT EXISTS (SELECT 1 FROM information_schema.tables "
            "WHERE table_schema = :s AND table_name = :t)"
        ), {"s": schema, "t": table}).scalar()


# ── Individual flag queries ─────────────────────────────────────────────────
# Each flag uses ST_Intersects on parcel_geom (full boundary, EPSG:5321).
# Returns one row per parcel with 0 or 1.

FLAG_QUERIES = {
    "flag_wetland": (
        "rea", "wetland",
        f"""
        SELECT p.parcel_id, 1 AS flag_wetland
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN rea.wetland w
          ON ST_Intersects(ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                           ST_Transform(w.geom, {CRS_ONTARIO}))
        GROUP BY p.parcel_id
        """
    ),
    "flag_woodland": (
        "rea", "woodland",
        f"""
        SELECT p.parcel_id, 1 AS flag_woodland
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN rea.woodland w
          ON ST_Intersects(ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                           ST_Transform(w.geom, {CRS_ONTARIO}))
        GROUP BY p.parcel_id
        """
    ),
    "flag_nhs": (
        "environment", "nhs_area",
        f"""
        SELECT p.parcel_id, 1 AS flag_nhs
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN environment.nhs_area n
          ON ST_Intersects(ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                           ST_Transform(n.geom, {CRS_ONTARIO}))
        GROUP BY p.parcel_id
        """
    ),
    "flag_saro_prox": (
        "environment", "prov_trk_species",
        f"""
        SELECT p.parcel_id, 1 AS flag_saro_prox
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN environment.prov_trk_species s
          ON ST_Intersects(ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                           ST_Transform(s.geom, {CRS_ONTARIO}))
        GROUP BY p.parcel_id
        """
    ),
    "flag_waa": (
        "environment", "wildlife_activity_area",
        f"""
        SELECT p.parcel_id, 1 AS flag_waa
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN environment.wildlife_activity_area w
          ON ST_Intersects(ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                           ST_Transform(w.geom, {CRS_ONTARIO}))
        GROUP BY p.parcel_id
        """
    ),
    "flag_flood": (
        "environment", "flood_hazard",
        f"""
        SELECT p.parcel_id, 1 AS flag_flood
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN environment.flood_hazard f
          ON ST_Intersects(ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                           ST_Transform(f.geom, {CRS_ONTARIO}))
        GROUP BY p.parcel_id
        """
    ),
    "flag_cli": (
        "environment", "cli",
        f"""
        SELECT p.parcel_id, 1 AS flag_cli
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN environment.cli c
          ON ST_Intersects(ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                           ST_Transform(c.geom, {CRS_ONTARIO}))
         AND c.cli_class IN (1, 2)
        GROUP BY p.parcel_id
        """
    ),
}

# Load parcel IDs to build a base DataFrame
df_base = pd.read_sql(
    f"SELECT parcel_id FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}",
    engine
)
df_flags = df_base.copy()

# Compute flag_ca_regulated from ca_overlap_ratio already in the table
df_ca_ratio = pd.read_sql(
    f"SELECT parcel_id, ca_overlap_ratio FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}",
    engine
)
df_flags = df_flags.merge(df_ca_ratio, on="parcel_id", how="left")
df_flags["flag_ca_regulated"] = (df_flags["ca_overlap_ratio"] > 0).astype(int)
df_flags = df_flags.drop(columns=["ca_overlap_ratio"])

# Spatial flags via PostGIS
for flag_col, (schema, table, sql) in FLAG_QUERIES.items():
    if _table_exists(schema, table):
        df_flag = pd.read_sql(sql, engine)
        df_flags = df_flags.merge(df_flag, on="parcel_id", how="left")
        df_flags[flag_col] = df_flags[flag_col].fillna(0).astype(int)
        present = df_flags[flag_col].sum()
        print(f"  {flag_col:20s}: {present:,} parcels flagged")
    else:
        df_flags[flag_col] = 0
        print(f"  {flag_col:20s}: table {schema}.{table} not found — defaulting to 0")

# ── ECI weighted sum ────────────────────────────────────────────────────────
ECI_WEIGHTS = {
    "flag_wetland": 3,
    "flag_woodland": 3,
    "flag_nhs": 2,
    "flag_saro_prox": 2,
    "flag_waa": 2,
    "flag_ca_regulated": 3,
    "flag_flood": 3,
    "flag_cli": 1,
}

df_flags["eci_raw"] = sum(
    df_flags[col] * weight for col, weight in ECI_WEIGHTS.items()
)

# env_flags JSON: human-readable flag summary per parcel
def _env_flags_json(row) -> str:
    return json.dumps({col: int(row[col]) for col in ECI_WEIGHTS})

df_flags["env_flags"] = df_flags.apply(_env_flags_json, axis=1)

print(f"\nStep 7 — ECI summary for '{COUNTY_NAME}':")
print(f"  Total parcels   : {len(df_flags):,}")
print(f"  ECI = 0 (clean) : {(df_flags['eci_raw'] == 0).sum():,}")
print(f"  ECI ≥ 10 (high) : {(df_flags['eci_raw'] >= 10).sum():,}")
print(f"  Mean ECI        : {df_flags['eci_raw'].mean():.2f} / 19")

flag_cols = list(ECI_WEIGHTS.keys()) + ["eci_raw", "env_flags"]
update_parcels_phase4(df_flags[["parcel_id"] + flag_cols])


---
## Step 8 — SARO Species Assessment

For each parcel, identify all SARO-listed species whose tracked range intersects
**`parcel_geom`**. Deduplicate by species name within each status category.

Output per parcel:
- `end_present` (boolean) — TRUE if any Endangered (END) species found
- `end_species` (text array) — list of END species names
- `thr_count` (integer) — unique Threatened (THR) species count
- `sc_count` (integer) — unique Special Concern (SC) species count
- `saro_raw` (integer) — `thr_count × 2 + sc_count × 1` (END handled separately in Phase 5)

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

has_saro = _table_exists("environment", "prov_trk_species")
print(f"SARO table 'environment.prov_trk_species' exists: {has_saro}")

if has_saro:
    df_saro_raw = pd.read_sql(f"""
        SELECT
            p.parcel_id,
            s.species_name,
            s.saro_status
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN environment.prov_trk_species s
          ON ST_Intersects(
                ST_Transform(p.parcel_geom, {CRS_ONTARIO}),
                ST_Transform(s.geom, {CRS_ONTARIO})
             )
        ORDER BY p.parcel_id, s.saro_status
    """, engine)
else:
    df_saro_raw = pd.DataFrame(columns=["parcel_id", "species_name", "saro_status"])
    print("  No SARO table — all parcels will have zero species counts")

# Aggregate per parcel
df_base = pd.read_sql(
    f"SELECT parcel_id FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}", engine
)

saro_agg = []
for parcel_id in df_base["parcel_id"]:
    df_p = df_saro_raw[df_saro_raw["parcel_id"] == parcel_id]

    end_names = df_p[df_p["saro_status"] == "END"]["species_name"].dropna().unique().tolist()
    thr_names = df_p[df_p["saro_status"] == "THR"]["species_name"].dropna().unique().tolist()
    sc_names  = df_p[df_p["saro_status"] == "SC"]["species_name"].dropna().unique().tolist()

    saro_agg.append({
        "parcel_id":   parcel_id,
        "end_present": len(end_names) > 0,
        "end_species": end_names,
        "thr_count":   len(thr_names),
        "sc_count":    len(sc_names),
        "saro_raw":    len(thr_names) * 2 + len(sc_names) * 1,
    })

df_saro = pd.DataFrame(saro_agg)
df_saro["end_species"] = df_saro["end_species"].apply(json.dumps)  # store as JSON string

print(f"Step 8 — SARO species assessment for '{COUNTY_NAME}':")
print(f"  Total parcels       : {len(df_saro):,}")
print(f"  Parcels with END    : {df_saro['end_present'].sum():,}")
print(f"  Parcels with THR    : {(df_saro['thr_count'] > 0).sum():,}")
print(f"  Parcels with SC     : {(df_saro['sc_count'] > 0).sum():,}")
print(f"  Max saro_raw        : {df_saro['saro_raw'].max()}")

update_parcels_phase4(df_saro)


---
## Step 9 — Triage Classification

Assign every parcel a traffic-light `triage_class` (GREEN / AMBER / RED) based
on the attributes computed in Steps 1–8. A structured `triage_flags` list records
every triggered condition.

**RED** (automatically excluded from scoring in Phase 5):
- `end_present = TRUE`
- PSW or ANSI intersects `buildable_geom`
- `ca_overlap_ratio >= 0.60`
- Grid capacity ratio < 0.5 (`(buildable_area_ac / 5.0) / grid_mw < 0.5`)

**AMBER** (scored but flagged — remediation note required):
- `ca_overlap_ratio > 0` and `< 0.60`
- THR or SC species present
- NHS intersects `buildable_geom`
- `slope_mean > 8`
- `noise_count_1km > 8`
- Grid capacity ratio between 0.5 and 0.7
- `noise_count_1km IS NULL` (data gap)

**GREEN**: none of the above triggered.

`data_completeness` summarises which environmental tiers were available.

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# ── Load full Phase 4 table for triage logic ────────────────────────────────
gdf_triage = read_postgis(
    f"SELECT * FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}",
    geom_col="buildable_geom"
)
df = gdf_triage.copy()

# ── PSW / ANSI check on buildable_geom ──────────────────────────────────────
has_sea = _table_exists("environment", "significant_ecological_area")
if has_sea:
    df_psw = pd.read_sql(f"""
        SELECT p.parcel_id, 1 AS psw_ansi_intersects
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN environment.significant_ecological_area sea
          ON ST_Intersects(
                ST_Transform(p.buildable_geom, {CRS_ONTARIO}),
                ST_Transform(sea.geom, {CRS_ONTARIO})
             )
        GROUP BY p.parcel_id
    """, engine)
    df = df.merge(df_psw, on="parcel_id", how="left")
    df["psw_ansi_intersects"] = df["psw_ansi_intersects"].fillna(0).astype(int)
else:
    df["psw_ansi_intersects"] = 0
    print("  environment.significant_ecological_area not found — PSW/ANSI flag set to 0")

# ── NHS check on buildable_geom ─────────────────────────────────────────────
has_nhs = _table_exists("environment", "nhs_area")
if has_nhs:
    df_nhs_build = pd.read_sql(f"""
        SELECT p.parcel_id, 1 AS nhs_on_buildable
        FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug} p
        JOIN environment.nhs_area n
          ON ST_Intersects(
                ST_Transform(p.buildable_geom, {CRS_ONTARIO}),
                ST_Transform(n.geom, {CRS_ONTARIO})
             )
        GROUP BY p.parcel_id
    """, engine)
    df = df.merge(df_nhs_build, on="parcel_id", how="left")
    df["nhs_on_buildable"] = df["nhs_on_buildable"].fillna(0).astype(int)
else:
    df["nhs_on_buildable"] = 0

# ── Grid capacity ratio ──────────────────────────────────────────────────────
df["grid_ratio"] = (df["buildable_area_ac"] / 5.0) / df["grid_mw"].replace(0, np.nan)

# ── Triage classification ────────────────────────────────────────────────────
def _classify(row) -> tuple[str, list[str]]:
    flags = []
    is_red = False

    # RED conditions
    if row.get("end_present", False):
        flags.append("END species present in parcel range")
        is_red = True

    if row.get("psw_ansi_intersects", 0):
        flags.append("PSW/ANSI intersects buildable area")
        is_red = True

    ca_ratio = float(row.get("ca_overlap_ratio", 0) or 0)
    if ca_ratio >= 0.60:
        flags.append(f"CA regulated area covers {ca_ratio:.0%} of buildable area (≥60%)")
        is_red = True

    grid_ratio = row.get("grid_ratio")
    grid_mw = row.get("grid_mw")
    if pd.notna(grid_ratio) and pd.notna(grid_mw) and grid_mw > 0 and grid_ratio < 0.5:
        flags.append(f"Grid capacity ratio {grid_ratio:.2f} < 0.5 (undersize for feeder)")
        is_red = True

    if is_red:
        return "RED", flags

    # AMBER conditions
    is_amber = False

    if 0 < ca_ratio < 0.60:
        flags.append(f"Partial CA regulated area overlap ({ca_ratio:.0%})")
        is_amber = True

    thr = int(row.get("thr_count", 0) or 0)
    sc  = int(row.get("sc_count", 0) or 0)
    if thr > 0 or sc > 0:
        flags.append(f"SARO species: {thr} THR, {sc} SC")
        is_amber = True

    if row.get("nhs_on_buildable", 0):
        flags.append("NHS overlaps buildable area")
        is_amber = True

    slope = row.get("slope_mean")
    if pd.notna(slope) and float(slope) > 8.0:
        flags.append(f"Mean slope {slope:.1f}% > 8%")
        is_amber = True

    noise = row.get("noise_count_1km")
    if pd.isna(noise):
        flags.append("Noise receptor count NULL (data gap)")
        is_amber = True
    elif int(noise) > 8:
        flags.append(f"Noise receptors within 1 km: {int(noise)} > 8")
        is_amber = True

    if pd.notna(grid_ratio) and 0.5 <= grid_ratio <= 0.7:
        flags.append(f"Grid capacity ratio {grid_ratio:.2f} between 0.5–0.7 (marginal)")
        is_amber = True

    if is_amber:
        return "AMBER", flags

    return "GREEN", []


results = df.apply(_classify, axis=1, result_type="expand")
df["triage_class"] = results[0]
df["triage_flags"] = results[1].apply(json.dumps)

# ── data_completeness ────────────────────────────────────────────────────────
provincial_layers  = ["flag_wetland", "flag_woodland", "flag_nhs", "flag_saro_prox",
                      "flag_flood", "flag_cli"]
county_layers      = ["flag_waa", "flag_ca_regulated"]
all_layers_present = all(c in df.columns for c in provincial_layers + county_layers)
prov_present       = all(c in df.columns for c in provincial_layers)

if all_layers_present and df["flag_waa"].any() or df["flag_ca_regulated"].any():
    has_multi_source = has_saro and has_sea
    completeness = "FULL_THREE_TIER" if has_multi_source else "PROVINCIAL_COUNTY"
elif prov_present:
    completeness = "PROVINCIAL_ONLY"
else:
    completeness = "PROVINCIAL_ONLY"

df["data_completeness"] = completeness

# ── Summary ──────────────────────────────────────────────────────────────────
counts = df["triage_class"].value_counts()
print(f"Step 9 — Triage classification for '{COUNTY_NAME}':")
print(f"  GREEN : {counts.get('GREEN', 0):,}")
print(f"  AMBER : {counts.get('AMBER', 0):,}")
print(f"  RED   : {counts.get('RED', 0):,}")
print(f"  data_completeness: {completeness}")

triage_cols = ["parcel_id", "triage_class", "triage_flags", "data_completeness"]
update_parcels_phase4(df[triage_cols])


---
## Step 10 — Verification & Interactive Map

Summary statistics and a Folium map coloured by `triage_class`.

Layers:
- **Parcels** coloured by triage class (GREEN / AMBER / RED)
- **REA exclusion zones** (context, semi-transparent)
- **Environmental flag overlay** (NHS, wetland, woodland — semi-transparent)

In [ ]:
county_slug = COUNTY_NAME.lower().replace(" ", "_")

# ── Load final Phase 4 table ─────────────────────────────────────────────────
gdf_final = read_postgis(
    f"""
    SELECT
        parcel_id,
        solar_parcel_uid,
        main_address,
        buildable_area_ac,
        grid_mw,
        feeder_voltage,
        slope_mean,
        pvout_mean,
        noise_count_1km,
        pp_ratio,
        ca_overlap_ratio,
        eci_raw,
        triage_class,
        triage_flags,
        data_completeness,
        flag_wetland, flag_woodland, flag_nhs, flag_saro_prox,
        flag_waa, flag_ca_regulated, flag_flood, flag_cli,
        end_present, saro_raw, thr_count, sc_count,
        parcel_geom AS geom
    FROM {OUTPUT_SCHEMA}.parcels_phase4_{county_slug}
    """,
    geom_col="geom"
)

# ── Summary stats ────────────────────────────────────────────────────────────
counts = gdf_final["triage_class"].value_counts()
print("=" * 55)
print(f"Phase 4 Summary — {COUNTY_NAME}")
print("=" * 55)
print(f"  Total parcels    : {len(gdf_final):,}")
print(f"  GREEN            : {counts.get('GREEN', 0):,}")
print(f"  AMBER            : {counts.get('AMBER', 0):,}")
print(f"  RED              : {counts.get('RED', 0):,}")
print(f"  Buildable area   : {gdf_final['buildable_area_ac'].sum():,.1f} ac total")
if gdf_final["slope_mean"].notna().any():
    print(f"  Mean slope       : {gdf_final['slope_mean'].mean():.1f} %")
if gdf_final["pvout_mean"].notna().any():
    print(f"  Mean PVout       : {gdf_final['pvout_mean'].mean():.1f} kWh/kWp/yr")
print(f"  Mean ECI raw     : {gdf_final['eci_raw'].mean():.2f} / 19")
print(f"  Data completeness: {gdf_final['data_completeness'].iloc[0]}")

# Triage trigger breakdown
print("\n  Top triage triggers:")
import itertools
all_flags = list(itertools.chain.from_iterable(
    json.loads(f) for f in gdf_final["triage_flags"] if f and f != "[]"
))
from collections import Counter
for trigger, n in Counter(all_flags).most_common(10):
    print(f"    {n:4d}x  {trigger}")

# ── Interactive map ──────────────────────────────────────────────────────────
TRIAGE_COLORS = {"GREEN": "#2e6b47", "AMBER": "#c77c0a", "RED": "#c0392b"}
TRIAGE_FILL   = {"GREEN": "#4caf7d", "AMBER": "#f0ae45", "RED": "#e57368"}

center = [gdf_final.geometry.centroid.y.mean(), gdf_final.geometry.centroid.x.mean()]

m = folium.Map(location=center, zoom_start=10, tiles=None)
MiniMap(toggle_display=True).add_to(m)

folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="Esri World Imagery",
    max_zoom=19,
).add_to(m)

# REA exclusion layer (context)
gdf_rea = read_postgis(
    f"SELECT geom FROM {OUTPUT_SCHEMA}.rea_exclusions_{county_slug}"
)
folium.GeoJson(
    gdf_rea.__geo_interface__,
    name="REA exclusion zones",
    style_function=lambda _: {
        "fillColor": "#89dd59", "color": "#0e4908",
        "weight": 0.5, "fillOpacity": 0.18
    }
).add_to(m)

# Parcels coloured by triage class
tooltip_fields = [
    "parcel_id", "main_address", "triage_class",
    "buildable_area_ac", "grid_mw", "feeder_voltage",
    "slope_mean", "pvout_mean", "noise_count_1km",
    "pp_ratio", "ca_overlap_ratio", "eci_raw",
    "end_present", "thr_count", "sc_count",
]
tooltip_aliases = [
    "Parcel ID", "Address", "Triage Class",
    "Buildable Area (ac)", "Grid Capacity (MW)", "Feeder Voltage (kV)",
    "Mean Slope (%)", "Mean PVout (kWh/kWp/yr)", "Buildings within 1 km",
    "Polsby-Popper Ratio", "CA Overlap Ratio", "ECI Raw",
    "END Species Present", "THR Count", "SC Count",
]
# Limit to columns actually present
present_pairs = [(f, a) for f, a in zip(tooltip_fields, tooltip_aliases)
                 if f in gdf_final.columns]
p_fields, p_aliases = zip(*present_pairs) if present_pairs else ([], [])

for triage_class in ["GREEN", "AMBER", "RED"]:
    subset = gdf_final[gdf_final["triage_class"] == triage_class]
    if subset.empty:
        continue
    color    = TRIAGE_COLORS[triage_class]
    fill_col = TRIAGE_FILL[triage_class]
    folium.GeoJson(
        subset.__geo_interface__,
        name=f"Parcels — {triage_class} ({len(subset):,})",
        style_function=lambda _, c=color, f=fill_col: {
            "fillColor": f, "color": c, "weight": 1.5, "fillOpacity": 0.7
        },
        highlight_function=lambda _, c=color: {
            "color": c, "weight": 3, "fillOpacity": 0.9
        },
        tooltip=folium.GeoJsonTooltip(
            fields=list(p_fields),
            aliases=list(p_aliases),
            localize=True, sticky=True
        )
    ).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

out_html = f"map_phase4_{county_slug}.html"
m.save(out_html)
print(f"\nSaved: {out_html}")
